# Per-Disease Binary Model Training

Trains one binary classifier (Healthy vs Disease) for each pathology defined in `src/config.py`.
Based on `model_training_v7.ipynb`.

In [ ]:
import sys
import json
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PowerTransformer, OneHotEncoder
from sklearn.feature_selection import VarianceThreshold
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay,
)
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

sys.path.append('..')
from src.config import PATHOLOGY_DE_TO_EN
from src.features import FeatureOptions, load_feature_tables
from src.sanity_check import run_all as sanity_check

In [ ]:
# Reproducibility & Config
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# Clinical gold standard: sustained vowel 'A' at normal pitch.
SELECTED_TOKEN = 'a_n'
MIN_SAMPLES_PER_CLASS = 10
MAX_SAMPLES_PER_CLASS = 500

# Downsample healthy to match total pathological count.
BALANCE_HEALTHY = True

# Data quality guards
EXCLUDE_OVERLAP_SPEAKERS = False
EXCLUDE_MIXED_BINARY_SPEAKERS = True

N_SPLITS = 5
THRESHOLD_GRID = np.linspace(0.35, 0.65, 11)

# Optimize for what you care about most
BINARY_THRESHOLD_OBJECTIVE = 'accuracy'  # 'accuracy' | 'balanced_accuracy'

# All diseases to train binary models for (from config.py)
# Keys are German pathology names; values are English labels.
# 'healthy' is excluded because it is the negative class for every model.
DISEASES = {
    de: en
    for de, en in PATHOLOGY_DE_TO_EN.items()
    if de != 'healthy'
}

print(f"Diseases loaded from config ({len(DISEASES)}):")
for de, en in DISEASES.items():
    print(f"  {de!r:40s} -> {en!r}")

opts = FeatureOptions(
    prefix=Path('..'),
    include_splits=True,
    random_seed=RANDOM_SEED,
    max_samples_per_class=MAX_SAMPLES_PER_CLASS,
    balance_healthy=BALANCE_HEALTHY,
    selected_token=SELECTED_TOKEN,
    num_workers=None,
    mfdfa_q_step=0.5,
    mfdfa_num_scales=40,
    target_sample_rate=50000,
)
opts

In [ ]:
# Load Features (Sample Level) with config-safety check
build_cfg_path = Path('..') / 'data' / 'processed' / 'features' / 'feature_build_config.json'
desired_cfg = {
    'max_samples_per_class': MAX_SAMPLES_PER_CLASS,
    'balance_healthy': BALANCE_HEALTHY,
    'target_sample_rate': opts.target_sample_rate,
    'selected_token': SELECTED_TOKEN,
}

force_rebuild = False
if build_cfg_path.exists():
    saved_cfg = json.loads(build_cfg_path.read_text(encoding='utf-8'))
    mismatches = {
        k: (saved_cfg.get(k), v)
        for k, v in desired_cfg.items()
        if saved_cfg.get(k) != v
    }
    if mismatches:
        force_rebuild = True
        print('Config mismatch vs cached features -> forcing rebuild:')
        for k, (old_v, new_v) in mismatches.items():
            print(f'  {k}: cached={old_v} -> desired={new_v}')

if force_rebuild:
    feature_dir = Path('..') / 'data' / 'processed' / 'features'
    for p in [
        feature_dir / 'sample_core.csv',
        feature_dir / 'acoustic_features.csv',
        feature_dir / 'multifractal_features.csv',
        feature_dir / 'opensmile_features.csv',
        feature_dir / 'sample_splits.csv',
        feature_dir / 'feature_summary.json',
        feature_dir / 'feature_build_config.json',
    ]:
        if p.exists():
            p.unlink()
    print('Deleted stale cached feature artifacts. Rebuilding now...')

tables = load_feature_tables(options=opts, build_if_missing=True, save_if_built=True)

core_df = tables['core'].drop_duplicates(subset=['sample_key']).copy()
acoustic_df = tables['acoustic'].drop_duplicates(subset=['sample_key']).copy()
multifractal_df = tables['multifractal'].drop_duplicates(subset=['sample_key']).copy()
opensmile_df = (
    tables.get('opensmile', pd.DataFrame()).drop_duplicates(subset=['sample_key']).copy()
    if 'opensmile' in tables and not tables.get('opensmile', pd.DataFrame()).empty
    else pd.DataFrame()
)
splits_df = (
    tables.get('splits', pd.DataFrame()).drop_duplicates(subset=['sample_key']).copy()
    if 'splits' in tables and not tables.get('splits', pd.DataFrame()).empty
    else pd.DataFrame()
)

df = core_df
df = df.merge(multifractal_df, on='sample_key', how='left')
if not opensmile_df.empty:
    df = df.merge(opensmile_df, on='sample_key', how='left')
if not splits_df.empty:
    df = df.merge(splits_df, on='sample_key', how='left')

if 'feature_status' in df.columns:
    df = df[df['feature_status'].isin(['ok', 'partial_failure'])].copy()
if 'acoustic_status' in df.columns:
    df = df[df['acoustic_status'] == 'ok'].copy()
if 'mf_status' in df.columns:
    df = df[df['mf_status'] == 'ok'].copy()
if 'opensmile_status' in df.columns:
    df = df[df['opensmile_status'] == 'ok'].copy()

# Compute age at recording
manifest_path = Path('..') / 'data' / 'processed' / 'manifests' / 'dataset_manifest.csv'
manifest_dates = pd.read_csv(manifest_path, usecols=['sample_key', 'birth_date', 'recording_date'])
manifest_dates = manifest_dates.drop_duplicates(subset=['sample_key'])
df = df.merge(manifest_dates, on='sample_key', how='left')
bd = pd.to_datetime(df['birth_date'], errors='coerce')
rd = pd.to_datetime(df['recording_date'], errors='coerce')
df['age'] = ((rd - bd).dt.days / 365.25).round(1)
df.drop(columns=['birth_date', 'recording_date'], inplace=True)
print(f"Age: mean={df['age'].mean():.1f}  min={df['age'].min():.1f}  max={df['age'].max():.1f}  missing={df['age'].isna().sum()}")

print('Merged sample-level shape:', df.shape)
print('Unique speakers:', df['speaker_id'].nunique())

In [ ]:
# Prep Feature Matrix
meta_cols = {
    'sample_key', 'duplicate_class_key', 'recording_id', 'speaker_id', 'wav_path',
    'feature_status', 'feature_error', 'acoustic_status', 'acoustic_error',
    'mf_status', 'mf_error', 'opensmile_status', 'opensmile_error',
    'split', 'split_seed', 'pathology_de', 'pathology_en', 'is_healthy',
    'is_overlap_speaker', 'is_overlap_speaker_id',
}

import re
_drop_patterns = [
    re.compile(r'mfcc', re.IGNORECASE),
    re.compile(r'F[123](frequency|bandwidth|amplitudeLogRelF0)', re.IGNORECASE),
    re.compile(r'lsp', re.IGNORECASE),
]

numeric_cols_all = [c for c in df.columns if c not in meta_cols and pd.api.types.is_numeric_dtype(df[c])]
_before = len(numeric_cols_all)
numeric_cols_all = [c for c in numeric_cols_all if not any(p.search(c) for p in _drop_patterns)]
print(f'Dropped {_before - len(numeric_cols_all)} features (MFCC, F1/F2/F3, LSP). Remaining: {len(numeric_cols_all)}')

# Drop highly collinear features (|r| > threshold). Keep the first in each correlated pair.
COLLINEARITY_THRESHOLD = 0.85
corr_matrix = df[numeric_cols_all].corr().abs()
upper_tri = corr_matrix.where(np.triu(np.ones(corr_matrix.shape, dtype=bool), k=1))
collinear_to_drop = [col for col in upper_tri.columns if any(upper_tri[col] > COLLINEARITY_THRESHOLD)]
if collinear_to_drop:
    print(f'Dropping {len(collinear_to_drop)} collinear features (|r| > {COLLINEARITY_THRESHOLD})')
    numeric_cols_all = [c for c in numeric_cols_all if c not in collinear_to_drop]

categorical_cols = ['sex'] if 'sex' in df.columns else []

print(f'Final feature count: {len(numeric_cols_all)} numeric + {len(categorical_cols)} categorical')

In [ ]:
# Preprocessing Pipeline
TREE_MODELS = (RandomForestClassifier, XGBClassifier, LGBMClassifier)


def make_preprocessors(numeric_cols, categorical_cols):
    """Build tree and linear preprocessors for the given feature columns."""
    if categorical_cols:
        tree_prep = ColumnTransformer(
            transformers=[
                ('num', 'passthrough', numeric_cols),
                ('cat', Pipeline(steps=[
                    ('imputer', SimpleImputer(strategy='most_frequent')),
                    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
                ]), categorical_cols),
            ],
            remainder='drop',
        ).set_output(transform='pandas')
        linear_prep = ColumnTransformer(
            transformers=[
                ('num', Pipeline(steps=[
                    ('imputer', SimpleImputer(strategy='constant', fill_value=0.0, add_indicator=True)),
                    ('scaler', PowerTransformer(method='yeo-johnson', standardize=True)),
                    ('variance', VarianceThreshold(threshold=0.0)),
                ]), numeric_cols),
                ('cat', Pipeline(steps=[
                    ('imputer', SimpleImputer(strategy='most_frequent')),
                    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
                ]), categorical_cols),
            ],
            remainder='drop',
        ).set_output(transform='pandas')
    else:
        tree_prep = ColumnTransformer(
            transformers=[('num', 'passthrough', numeric_cols)],
            remainder='drop',
        ).set_output(transform='pandas')
        linear_prep = ColumnTransformer(
            transformers=[('num', Pipeline(steps=[
                ('imputer', SimpleImputer(strategy='constant', fill_value=0.0, add_indicator=True)),
                ('scaler', PowerTransformer(method='yeo-johnson', standardize=True)),
                ('variance', VarianceThreshold(threshold=0.0)),
            ]), numeric_cols)],
            remainder='drop',
        ).set_output(transform='pandas')
    return tree_prep, linear_prep


def make_pipe(model, tree_prep, linear_prep):
    """Build pipeline: auto-selects tree vs linear preprocessor."""
    prep = tree_prep if isinstance(model, TREE_MODELS) else linear_prep
    return Pipeline(steps=[('prep', prep), ('model', model)])

In [ ]:
def evaluate_binary_cv(model, X, y, groups, tree_prep, linear_prep, n_splits=5):
    """Stratified group K-fold binary CV with threshold tuning on train probas."""
    cv = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_SEED)
    fold_rows = []
    all_y_true, all_y_pred = [], []
    for fold, (tr, te) in enumerate(cv.split(X, y, groups), start=1):
        X_tr, X_te = X.iloc[tr], X.iloc[te]
        y_tr, y_te = y.iloc[tr], y.iloc[te]

        pipe = make_pipe(clone(model), tree_prep, linear_prep)
        pipe.fit(X_tr, y_tr)

        # Threshold tuning on train probas
        p_tr = pipe.predict_proba(X_tr)[:, 1]
        best_thr, best_score = 0.5, -1.0
        for thr in THRESHOLD_GRID:
            pred_tr = (p_tr >= thr).astype(int)
            score = (
                accuracy_score(y_tr, pred_tr)
                if BINARY_THRESHOLD_OBJECTIVE == 'accuracy'
                else balanced_accuracy_score(y_tr, pred_tr)
            )
            if score > best_score:
                best_score = score
                best_thr = float(thr)

        p_te = pipe.predict_proba(X_te)[:, 1]
        pred_te = (p_te >= best_thr).astype(int)
        all_y_true.extend(y_te.tolist())
        all_y_pred.extend(pred_te.tolist())

        fold_rows.append({
            'fold': fold,
            'threshold': best_thr,
            'accuracy': accuracy_score(y_te, pred_te),
            'balanced_accuracy': balanced_accuracy_score(y_te, pred_te),
            'f1_macro': f1_score(y_te, pred_te, average='macro', zero_division=0),
        })

    fold_df = pd.DataFrame(fold_rows)
    return fold_df, {
        'accuracy_mean': fold_df['accuracy'].mean(),
        'balanced_accuracy_mean': fold_df['balanced_accuracy'].mean(),
        'f1_macro_mean': fold_df['f1_macro'].mean(),
    }, np.array(all_y_true), np.array(all_y_pred)

In [ ]:
# Per-Disease Binary Model Training
# For each disease in PATHOLOGY_DE_TO_EN, train binary classifiers:
#   positive class (1) = Healthy
#   negative class (0) = Disease

binary_models = {
    'LogReg': LogisticRegression(
        max_iter=3000, class_weight='balanced', C=1.0, random_state=RANDOM_SEED,
    ),
    'RandomForest': RandomForestClassifier(
        n_estimators=800, max_depth=None, min_samples_leaf=2,
        class_weight='balanced_subsample', random_state=RANDOM_SEED, n_jobs=-1,
    ),
    'LightGBM': LGBMClassifier(
        n_estimators=800, learning_rate=0.03, num_leaves=63,
        min_child_samples=15, subsample=0.8, colsample_bytree=0.8,
        class_weight='balanced', random_state=RANDOM_SEED, n_jobs=-1, verbose=-1,
    ),
    'XGBoost': XGBClassifier(
        n_estimators=700, learning_rate=0.03, max_depth=6,
        subsample=0.8, colsample_bytree=0.8, min_child_weight=3,
        reg_alpha=1.0, reg_lambda=2.0,
        random_state=RANDOM_SEED, n_jobs=-1, eval_metric='logloss',
    ),
}

# Collect per-disease results: {disease_en: {model_name: (fold_df, summ, yt, yp)}}
all_disease_results = {}
summary_rows = []

# Remove mixed binary speakers (healthy + pathological) from the full frame once.
df_clean = df.copy()
if EXCLUDE_MIXED_BINARY_SPEAKERS and 'speaker_id' in df_clean.columns and 'is_healthy' in df_clean.columns:
    grp = df_clean.groupby(df_clean['speaker_id'].astype(str))['is_healthy'].nunique()
    bad_speakers = set(grp[grp > 1].index.tolist())
    if bad_speakers:
        removed = int(df_clean['speaker_id'].astype(str).isin(bad_speakers).sum())
        print(f'Removing mixed-label speaker rows: {removed} from {len(bad_speakers)} speakers')
        df_clean = df_clean.loc[~df_clean['speaker_id'].astype(str).isin(bad_speakers)].copy()

healthy_df = df_clean[df_clean['is_healthy'].astype(bool)].copy()

for disease_de, disease_en in DISEASES.items():
    # Select samples for this disease (match on German pathology name)
    disease_mask = df_clean['pathology_de'].astype(str).str.strip() == disease_de
    disease_df = df_clean[disease_mask].copy()

    n_disease = len(disease_df)
    if n_disease < MIN_SAMPLES_PER_CLASS:
        print(f'  Skipping {disease_en!r}: only {n_disease} samples (< {MIN_SAMPLES_PER_CLASS})')
        continue

    # Balance: use at most n_disease healthy samples
    n_healthy_use = min(len(healthy_df), n_disease)
    healthy_sub = healthy_df.sample(n=n_healthy_use, random_state=RANDOM_SEED)

    sub_df = pd.concat([healthy_sub, disease_df], ignore_index=True)
    y_bin = sub_df['is_healthy'].astype(int).copy()
    groups = sub_df['speaker_id'].astype(str).copy()

    # Restrict numeric_cols to those present after column filtering
    nc = [c for c in numeric_cols_all if c in sub_df.columns]
    cc = [c for c in categorical_cols if c in sub_df.columns]
    X_sub = sub_df[nc + cc].copy()

    n_classes_in_groups = sub_df.groupby(groups)['is_healthy'].nunique()
    if (n_classes_in_groups > 1).sum() < N_SPLITS:
        actual_splits = max(2, (n_classes_in_groups > 1).sum())
        print(f'  {disease_en!r}: reducing CV splits to {actual_splits} (not enough mixed-class speakers)')
    else:
        actual_splits = N_SPLITS

    tree_prep, linear_prep = make_preprocessors(nc, cc)

    print(f'\n=== {disease_en!r} (n={len(sub_df)}: {n_disease} disease + {n_healthy_use} healthy) ===')
    disease_model_results = {}
    for model_name, model in binary_models.items():
        try:
            fold_df, summ, yt, yp = evaluate_binary_cv(
                model, X_sub, y_bin, groups, tree_prep, linear_prep, n_splits=actual_splits,
            )
            disease_model_results[model_name] = (fold_df, summ, yt, yp)
            print(f'  {model_name:15s}  acc={summ["accuracy_mean"]:.3f}  '
                  f'bal_acc={summ["balanced_accuracy_mean"]:.3f}  '
                  f'f1={summ["f1_macro_mean"]:.3f}')
            summary_rows.append({
                'disease': disease_en,
                'n_disease': n_disease,
                'n_healthy': n_healthy_use,
                'model': model_name,
                **summ,
            })
        except Exception as exc:
            print(f'  {model_name:15s}  ERROR: {exc}')
    all_disease_results[disease_en] = disease_model_results

In [ ]:
# Summary Table: Best Model per Disease
if summary_rows:
    summary_df = pd.DataFrame(summary_rows)
    best_per_disease = (
        summary_df
        .sort_values('balanced_accuracy_mean', ascending=False)
        .groupby('disease', sort=False)
        .first()
        .reset_index()
        .sort_values('balanced_accuracy_mean', ascending=False)
    )
    print('Best model per disease (by balanced accuracy):')
    display(best_per_disease[['disease', 'n_disease', 'n_healthy', 'model',
                               'accuracy_mean', 'balanced_accuracy_mean', 'f1_macro_mean']])

    print('\nAll results:')
    display(
        summary_df
        .sort_values(['disease', 'balanced_accuracy_mean'], ascending=[True, False])
        .reset_index(drop=True)
    )
else:
    print('No results collected — all diseases were skipped due to insufficient samples.')

In [ ]:
# Confusion Matrices per Disease (best model)
for disease_en, model_results in all_disease_results.items():
    if not model_results:
        continue
    fig, axes = plt.subplots(1, len(model_results), figsize=(5 * len(model_results), 4))
    if len(model_results) == 1:
        axes = [axes]
    for ax, (model_name, (_, _, yt, yp)) in zip(axes, model_results.items()):
        cm = confusion_matrix(yt, yp, labels=[0, 1])
        ConfusionMatrixDisplay(cm, display_labels=['Disease', 'Healthy']).plot(
            ax=ax, cmap='Blues', colorbar=False,
        )
        ax.set_title(model_name)
    fig.suptitle(f'{disease_en} — Confusion Matrices (CV-aggregated)', y=1.02)
    fig.tight_layout()
    plt.show()

In [ ]:
# Feature Importances per Disease (tree models, retrained on full subset)
TREE_MODEL_NAMES = {name for name, m in binary_models.items() if isinstance(m, TREE_MODELS)}


def _get_feature_names(pipe):
    return pipe.named_steps['prep'].get_feature_names_out()


for disease_de, disease_en in DISEASES.items():
    if disease_en not in all_disease_results or not all_disease_results[disease_en]:
        continue

    disease_mask = df_clean['pathology_de'].astype(str).str.strip() == disease_de
    disease_df = df_clean[disease_mask].copy()
    n_healthy_use = min(len(healthy_df), len(disease_df))
    healthy_sub = healthy_df.sample(n=n_healthy_use, random_state=RANDOM_SEED)
    sub_df = pd.concat([healthy_sub, disease_df], ignore_index=True)
    y_bin = sub_df['is_healthy'].astype(int).copy()
    nc = [c for c in numeric_cols_all if c in sub_df.columns]
    cc = [c for c in categorical_cols if c in sub_df.columns]
    X_sub = sub_df[nc + cc].copy()
    tree_prep, linear_prep = make_preprocessors(nc, cc)

    tree_model_results = [
        (name, m)
        for name, m in binary_models.items()
        if name in TREE_MODEL_NAMES and name in all_disease_results[disease_en]
    ]
    if not tree_model_results:
        continue

    fig, axes = plt.subplots(
        len(tree_model_results), 1,
        figsize=(12, 5 * len(tree_model_results)),
        squeeze=False,
    )
    for ax, (model_name, model) in zip(axes[:, 0], tree_model_results):
        pipe = make_pipe(clone(model), tree_prep, linear_prep)
        pipe.fit(X_sub, y_bin)
        importances = pipe.named_steps['model'].feature_importances_
        feat_names = _get_feature_names(pipe)
        top_k = min(20, len(feat_names))
        sorted_idx = np.argsort(importances)[-top_k:]
        ax.barh(range(top_k), importances[sorted_idx], color='steelblue')
        ax.set_yticks(range(top_k))
        ax.set_yticklabels([feat_names[i] for i in sorted_idx])
        ax.set_title(f'{disease_en} — {model_name} (top {top_k})')
        ax.set_xlabel('Importance')
    fig.suptitle(f'{disease_en} — Feature Importances (top 20, trained on full subset)', y=1.01)
    fig.tight_layout()
    plt.show()